# Quaternions and Euler Angles

This notebook covers quaternions and euler angle theory needed for an Extended Kalman Filter (EKF) for state estimation.


**References:**
- Groves, P. D. (2013). *Principles of GNSS, Inertial, and Multi-Sensor Integrated Navigation Systems*, 2nd ed. Artech House. [ProQuest link](https://ebookcentral.proquest.com/lib/fh-ostschweiz/reader.action?docID=1531533&c=UERG&ppg=1)
- YouTube: (3Blue1Brown)[https://www.youtube.com/watch?v=d4EgbgTm0Bg&t=371s]
- Lisyarus blog: [(Yet another) Introduction to quaternions](https://lisyarus.github.io/blog/posts/introduction-to-quaternions.html)

Additionally, more about rotation handling by rocketpy can be found [here](https://docs.rocketpy.org/en/latest/user/rocket/rocket_axes.html).

These first few functions are used to define plots that visualize the topics.

In [ ]:
import numpy as np
import plotly.graph_objects as go


def _add_arrow(fig, origin, direction, color, name, width=6, head_size=0.12, showlegend=True):
    end = origin + direction
    fig.add_trace(go.Scatter3d(
        x=[origin[0], end[0]],
        y=[origin[1], end[1]],
        z=[origin[2], end[2]],
        mode="lines",
        line=dict(color=color, width=width),
        name=name,
        showlegend=showlegend,
    ))
    fig.add_trace(go.Cone(
        x=[end[0]], y=[end[1]], z=[end[2]],
        u=[direction[0]], v=[direction[1]], w=[direction[2]],
        sizemode="absolute",
        sizeref=head_size,
        anchor="tip",
        colorscale=[[0, color], [1, color]],
        showscale=False,
        showlegend=False,
    ))


def plot_orientation(R, title="", show_rocket=True, extra_vectors=None):
    # R: 3x3 rotation matrix (body -> world). A body-frame vector v_b is drawn
    # in world coordinates as R @ v_b.
    # Body: +z_b along centerline toward nose, x_b/y_b perpendicular (rocketpy convention)
    # extra_vectors: list of (v_body, color, name) to also draw, rotated by R.
    fig = go.Figure()

    # World frame
    for axis, label in zip(np.eye(3), ["x_w", "y_w", "z_w"]):
        _add_arrow(fig, np.zeros(3), 0.7 * axis, color="lightgrey", name=label, width=3)

    # Body frame:
    for axis, color, name in [
        (np.array([1.0, 0.0, 0.0]), "red", "x_b"),
        (np.array([0.0, 1.0, 0.0]), "green", "y_b"),
        (np.array([0.0, 0.0, 1.0]), "blue", "z_b (nose)"),
    ]:
        _add_arrow(fig, np.zeros(3), R @ axis, color=color, name=name, width=7)

    if show_rocket:
        # Rocket body runs along +z_b (centerline, nose tip at +z_b)
        nose = R @ np.array([0.0, 0.0, 1.2])
        fig.add_trace(go.Scatter3d(
            x=[0, nose[0]], y=[0, nose[1]], z=[0, nose[2]],
            mode="lines",
            line=dict(color="slategray", width=18),
            name="rocket body",
            opacity=0.6,
        ))

    if extra_vectors is not None:
        for v_body, color, name in extra_vectors:
            _add_arrow(fig, np.zeros(3), R @ np.asarray(v_body, dtype=float),
                       color=color, name=name, width=5)

    fig.update_layout(
        title=title,
        scene=dict(
            xaxis=dict(range=[-1.5, 1.5], title="x (world)"),
            yaxis=dict(range=[-1.5, 1.5], title="y (world)"),
            zaxis=dict(range=[-1.5, 1.5], title="z (world)"),
            aspectmode="cube",
            camera=dict(eye=dict(x=1.6, y=1.6, z=1.0)),
        ),
        legend=dict(itemsizing="constant"),
        margin=dict(l=0, r=0, t=40, b=0),
        height=500,
    )
    return fig


# body frame aligned with world frame, rocket points up along +z
plot_orientation(np.eye(3), title="Identity rotation").show()

## Introduction

Attitude describes how a body is oriented in 3D space. For the rocket, the on-board IMU measures angular rates and specific force in the body frame (b-frame), whose axes are bolted to the rocket and rotate with it. GNSS, by contrast, delivers position and velocity in the local tangent-plane frame (l-frame). To fuse these two sensor streams, every b-frame measurement must be rotated into the l-frame, which requires knowing the current attitude. Because attitude changes continuously during flight, the Kalman filter has to carry it as part of its state and propagate it forward using gyro measurements.

Generally, the EKF carries a parameterization of the rotation (either euler angles or quaternions) and the matrix $C^l_b$ can be constructed from it. This matrix can then be applied to vector to rotate them. Note: for quaternions, the matrix does not necessarily need to be constructed.

### Attitude vs. coordinate transformation

Two closely related concepts appear throughout the rotation literature:

**Attitude** is the rotation of an object's orientation relative to a fixed set of resolving axes. When the rocket pitches over by 10°, its *body* rotates while the reference frame stays the same.

**Coordinate transformation** is the rotation of the resolving axes relative to the object. The *same* physical situation can be described by saying the axes rotated by 10° in the opposite direction while the body stayed the same.

These two operations are inverses of each other: rotating the body by $R$ is equivalent to rotating the frame by $R^\top$. For a rotation matrix, the inverse equals the transpose, so $C^\beta_\alpha = (C^\alpha_\beta)^\top$ (Groves eq. 2.14)


Throughout this notebook, rotations are written as coordinate transformations (using a coordinate transformation matrix) of the form $C^\beta_\alpha$, read as *"takes a vector expressed in frame $\alpha$ and re-expresses it in frame $\beta$."*

$$
\mathbf{v}^\beta = C^\beta_\alpha\, \mathbf{v}^\alpha
$$

Indices chain as follows: $C^l_b = C^l_e\, C^e_b$, and inverses flip the indices: $C^b_l = (C^l_b)^\top$.


## Euler Attitude (roll, pitch, yaw)

Euler angles are the most intuitive way to describe an attitude. They are three successive rotations about defined axes, applied in a fixed order (can be freely defined).

Groves applies them in order yaw -> pitch -> roll, which corresponds to the `"zyx"` convention in `scipy`. The angles are conventionally listed in the reverse order to which they are applied:

$$
\boldsymbol{\psi}^l_b = \begin{pmatrix} \phi^l_b \\ \theta^l_b \\ \psi^l_b \end{pmatrix}
$$

These angles can then be used to construct a rotation matrix using the formula (2.22) from Groves.

### Disadvantage of Euler angles

Euler angles are intuitive but hava a few drawbacks:
- **Reversing**: reversing a rotation can not be done by negating the angles. It has to be done via the rotation matrix
- **Composing rotations**: chaining multiple rotations is also not possible with only the angles. Gyro increments therefore cannot be accumulated directly in Euler form
- **Gimbal lock**: if two axes align, one degree of freedom is lost.


For these reasons the EKF should mainly work with quaternions.

In [ ]:
from scipy.spatial.transform import Rotation as R

rocket_attitude = R.from_euler("zyx", [0, 0, 20], degrees=True).as_matrix()
plot_orientation(rocket_attitude, title="Euler Rotation").show()

## Quaternion Attitude

A quaternion is a 4-parameter description of a rotation:

$$
\mathbf{q}^l_b = \begin{pmatrix} q_w \\ q_x \\ q_y \\ q_z \end{pmatrix}
$$

where $q_w$ is the scalar part and $(q_x, q_y, q_z)$ is the vector part. Note that there are representations working with scalar first or scalar last. This notebook will use scalar first notation.

For the quaternion to represent a valid rotation it must have unit norm:

$$
q_w^2 + q_x^2 + q_y^2 + q_z^2 = 1
$$

### Geometric meaning

Any 3D rotation can be expressed as a rotation by some angle $\alpha$ around some unit axis $\mathbf{n} = (n_x, n_y, n_z)$. The corresponding quaternion is:

$$
\mathbf{q} = \begin{pmatrix} \cos(\alpha/2) \\ n_x \sin(\alpha/2) \\ n_y \sin(\alpha/2) \\ n_z \sin(\alpha/2) \end{pmatrix}
$$

### Quaternion multiplication (Hamilton product)

Composing two rotations means [multiplying their quaternions](https://ch.mathworks.com/help/aeroblks/quaternionmultiplication.html). For $\mathbf{p} = (p_w, p_x, p_y, p_z)$ and $\mathbf{q} = (q_w, q_x, q_y, q_z)$:

$$
\mathbf{p} \otimes \mathbf{q} = \begin{pmatrix}
p_w q_w - p_x q_x - p_y q_y - p_z q_z \\
p_w q_x + p_x q_w + p_y q_z - p_z q_y \\
p_w q_y - p_x q_z + p_y q_w + p_z q_x \\
p_w q_z + p_x q_y - p_y q_x + p_z q_w
\end{pmatrix}
$$

### Inverse of a quaternion

For a unit quaternion, the inverse equals the conjugate (negate the vector part):

$$
\mathbf{q}^{-1} = \mathbf{q}^* = (q_w, -q_x, -q_y, -q_z)
$$

This is the quaternion equivalent of $C^\beta_\alpha = (C^\alpha_\beta)^\top$: if $\mathbf{q}^l_b$ rotates body vectors into NED, then $(\mathbf{q}^l_b)^*$ rotates NED vectors into body.

### Applying a quaternion to a vector

A vector $\mathbf{v}^b$ is rotated into the l-frame using the following formula:

$$
\begin{pmatrix} 0 \\ \mathbf{v}^l \end{pmatrix} = \mathbf{q}^l_b \otimes \begin{pmatrix} 0 \\ \mathbf{v}^b \end{pmatrix} \otimes (\mathbf{q}^l_b)^*
$$

Alternatively, the matrix $C^l_b$ can be constructed from the quaternion $\mathbf{q}^l_b$ and applied to the vector $\mathbf{v}^b$ (Groves, 2.34), although this is generally slower.

Input from Claude: In practice, when many vectors need to be rotated with the same $\mathbf{q}^l_b$ — or when the Jacobian of the rotation is needed for the EKF — it is more efficient to build $C^l_b$ once and apply it as a matrix-vector product $\mathbf{v}^l = C^l_b\, \mathbf{v}^b$. The conversion $\mathbf{q} \to C^l_b$ is covered in the next section.


A few additional notes:

- **Not commutative**: $\mathbf{p} \otimes \mathbf{q} \neq \mathbf{q} \otimes \mathbf{p}$ in general
- **Identity**: $\mathbf{q}_I = (1, 0, 0, 0)$ represents no rotation. Note that this is also scalar first!
- **Inverse**: for a unit quaternion, $\mathbf{q}^{-1} = \mathbf{q}^* = (q_w, -q_x, -q_y, -q_z)$.
- **Composition order**: $\mathbf{q}^l_b = \mathbf{q}^l_e \otimes \mathbf{q}^e_b$ chains indices the same way rotation matrices do.


In [ ]:
euler_attitude = R.from_euler("zyx", [0, 0, 20], degrees=True)
quaternion_attitude = euler_attitude.as_quat(scalar_first=True)
rocket_attitude = R.from_quat(quaternion_attitude, scalar_first=True).as_matrix()

plot_orientation(rocket_attitude, title="Quaternion Rotation").show()

### Quaternion updates

Quaternions also allow for updates of increments. This is useful if a new sensor reading is available and should be incorporated into the current attitude. The following equation describes accumulation of small rotations into the total attitude of the rocket:

$$
\mathbf{q}_b^l(t + \Delta t) = q_b^l \otimes \Delta q_b
$$

$\mathbf{q}^l_b$: rocket's attitude, rotating vectors from the body frame to the NED frame \
$\Delta \mathbf{q}_b$: incremental rotation expressed in the body frame

The increment is right-multiplied because it is expressed in the body frame: the rotation $\Delta \mathbf{q}_b$ must act on a vector *before* $\mathbf{q}^l_b$ transforms it into the NED frame. A rotation increment expressed in the NED frame would instead be left-multiplied.

In [ ]:
dt = 2  # seconds
q = R.from_euler("zyx", [0, 0, 10], degrees=True)  # initial orientation staight up

gyro_measurement = np.array([0.0, 0.0, np.deg2rad(10)])  # gyro reading, 10 deg/s about body-z
actual_rotation = gyro_measurement * dt

delta_q = R.from_rotvec(actual_rotation)

q_lb_new = q * delta_q  # body frame rate, thus right multiply!!!!

plot_orientation(q_lb_new.as_matrix(), title="Quaternion Update Rotation").show()
print("Old quaternion (x,y,z,w):", q.as_quat(scalar_first=True))
print("New quaternion (x,y,z,w):", q_lb_new.as_quat(scalar_first=True))
print("New Euler (zyx, deg):   ", q_lb_new.as_euler("zyx", degrees=True))

## Additional topics worth noting

- **Quaternion double cover**: Every rotation has two quaternion representations, q and -q describe the same attitude
- **Normalization drift**: always normalize because small rotation increments over time break the unit norm constraint
- **Gyro bias**: a bias term is almost always part of an EKF state vector for attitude
- **Error state**: For attitude, EKFs almost always use an error-state rather than putting the four quaternion components directly in the state vector.